[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C00_VLM_Multimodal_Course/05_instruction_tuning/05_instruction_tuning.ipynb)

# 05 · 视觉指令微调与对齐 (Visual Instruction Tuning & Alignment)

**配套讲解**：`05_讲解.html` · **算力**：需要 GPU。基座以 **4-bit (NF4)** 量化加载，配合 LoRA，约 **8–12GB 显存**即可（单张消费级卡可跑）。

本 notebook 动手验证讲解里的核心机制，全部用真实 HuggingFace API、真实模型权重：

1. 4-bit 量化加载小 VLM **`Qwen/Qwen2-VL-2B-Instruct`**（`Qwen2VLForConditionalGeneration` + `AutoProcessor`）
2. 构造一个极小视觉指令数据集，用 processor 的 **chat 模板**格式化
3. 用 **PEFT / LoRA** 包模型，打印可训练参数占比（trainable %）
4. 一个**演示规模**的训练循环：前向得 loss、反向、AdamW step，画 loss 曲线
5. **before / after**：对留出样例生成，对比微调前后回答
6. **DPO 概念演示**：给一个偏好对，按 DPO loss 公式手算一个 batch 的 loss，讲清隐式奖励
7. **3 道 ✏️ 练习**（纯 CPU 可跑，不依赖上面的 GPU cell）：label mask、LoRA 参数量、numpy 手写 sequence log-prob

> ⚠️ 这是**教学演示**：数据极小、step 极少，目的是看清*机制与 API*，不是训出一个好模型。真实微调需要成千上万条数据、完整的训练框架（如 TRL 的 `SFTTrainer` / `DPOTrainer`）。

In [ ]:
# ── 导入 + 设备 + 版本 ───────────────────────────────────────────────
import torch
import transformers, peft
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

# 4-bit 量化依赖 CUDA（bitsandbytes）；本 notebook 设计在 GPU 上运行
device = "cuda" if torch.cuda.is_available() else "cpu"

print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("peft         :", peft.__version__)
print("device       :", device)
if device != "cuda":
    print("\n[注意] 未检测到 CUDA。4-bit 量化 (bitsandbytes) 需要 NVIDIA GPU。"
          "\n        在 CPU/MPS 上本 notebook 的量化加载与训练将无法正常运行。")

## 1. 4-bit (NF4) 量化加载小 VLM

我们用 **`BitsAndBytesConfig`** 把 2B 基座量化到 4-bit，这正是讲解第 3 节 **QLoRA** 的配置：

- `load_in_4bit=True`：基座权重存为 4-bit
- `bnb_4bit_quant_type="nf4"`：用 **NF4**（为近似正态分布的权重设计的 4-bit 类型）
- `bnb_4bit_use_double_quant=True`：**双重量化**（连量化常数也量化，再省 ~0.4 bit/参数）
- `bnb_4bit_compute_dtype=torch.bfloat16`：前向时把 4-bit 权重临时反量化回 bf16 参与计算

> 模型 + processor 首次会从 HuggingFace 下载（约 4–5GB）。`Qwen2-VL-2B-Instruct` 本身已是 instruct 模型，这里只是借它演示*在其之上再做 LoRA SFT / DPO* 的完整流程。

In [ ]:
# ── 4-bit 量化加载基座 ────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

# 4-bit (bitsandbytes) 需要 CUDA；包 try/except 以保证无 GPU/显存不足/无网时优雅降级，
# 后续训练/生成 cell 会自动跳过真实模型部分（练习均为纯 Python，不受影响），端到端不报错。
model = processor = None
MODEL_OK = False
try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",          # NF4：信息论最优的 4-bit 正态量化
        bnb_4bit_use_double_quant=True,     # 双重量化
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    processor = AutoProcessor.from_pretrained(MODEL_ID)

    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    model.config.use_cache = False  # 训练时关掉 KV cache
    MODEL_OK = True

    print("模型加载完成（4-bit）。")
    print("约占显存 (GB):", round(model.get_memory_footprint() / 1e9, 2))
except Exception as e:
    print(f"[优雅降级] 未能 4-bit 加载基座（{type(e).__name__}: {str(e)[:160]}）。")
    print("  4-bit 量化 + LoRA 训练需要 NVIDIA GPU；本 notebook 的 3 个练习（label mask / "
          "LoRA 参数量 / 序列 log-prob）是纯 Python，仍可完整完成。")


## 2. 构造极小视觉指令数据集 + chat 模板格式化

视觉指令数据的基本单元是 **(图像 image, 指令 instruction, 目标回答 target)** 三元组（讲解第 2 节）。
下面手写 4 条（涵盖讲解里的 conversation / detail / reasoning 三类），图像用公网 url。

关键：用 **processor 的 chat 模板**把"用户问 + 助手答"拼成模型期望的对话格式。
`apply_chat_template(..., add_generation_prompt=False)` 会把<i>包含助手回答</i>的完整对话渲染成训练序列。

In [ ]:
# ── 极小视觉指令数据集（图像 url + 指令 + 目标回答）──────────────────
CAT_URL = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"

train_samples = [
    # conversation：直接读图的事实问答
    {"image": CAT_URL,
     "instruction": "What animal is in this image? Answer in one word.",
     "target": "Dog."},
    # detail description：详细描述
    {"image": CAT_URL,
     "instruction": "Describe this image in one sentence.",
     "target": "A woman sits on the beach at sunset, interacting with her dog."},
    # conversation：场景属性
    {"image": CAT_URL,
     "instruction": "What is the setting of this scene?",
     "target": "A sandy beach near the ocean during sunset."},
    # complex reasoning：推断意图
    {"image": CAT_URL,
     "instruction": "Why might the person and the dog be raising their hands/paw?",
     "target": "They appear to be playfully giving each other a high-five, suggesting a friendly bonding moment."},
]

# 留出一个样例做 before/after 对比（不参与训练）
heldout_sample = {
    "image": CAT_URL,
    "instruction": "What is the mood of this image? Answer briefly.",
}

from PIL import Image
import requests

def load_image(url):
    return Image.open(requests.get(url, stream=True).raw).convert("RGB")

def build_inputs(sample, with_target):
    '''用 chat 模板把 (图, 指令[, 回答]) 渲染成模型输入。
    with_target=True 用于训练（含助手回答）；False 用于生成（只到 generation prompt）。'''
    content = [
        {"type": "image"},
        {"type": "text", "text": sample["instruction"]},
    ]
    messages = [{"role": "user", "content": content}]
    if with_target:
        messages.append({"role": "assistant",
                         "content": [{"type": "text", "text": sample["target"]}]})
    text = processor.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=not with_target,   # 生成时补 <|assistant|> 提示
    )
    image = load_image(sample["image"])
    inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)
    return inputs

# 看一眼渲染出的训练文本
if MODEL_OK:
    demo_text = processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "image"},
                                      {"type": "text", "text": train_samples[0]["instruction"]}]},
         {"role": "assistant", "content": [{"type": "text", "text": train_samples[0]["target"]}]}],
        tokenize=False, add_generation_prompt=False)
    print(demo_text)
else:
    print("[跳过] processor 未加载，无法渲染 chat 模板示例。")


## 3. 用 PEFT / LoRA 包模型，打印可训练参数占比

这是讲解第 3 节 **LoRA** 的代码落地：把低秩增量 $\Delta W=\frac{\alpha}{r}BA$ 挂到 LLM 的注意力与 MLP 投影层上。

- `target_modules`：注意力投影 `q/k/v/o_proj` + MLP `gate/up/down_proj`（Qwen2-VL 的 LLM 模块名）
- `r=8` / `lora_alpha=16`：秩与缩放（$\alpha/r=2$）
- `prepare_model_for_kbit_training`：4-bit 训练必备（开梯度检查点、把 LayerNorm 等转 fp32 保稳定）

打印 **trainable %**，直观看到"只训练零点几个百分点的参数"。

In [ ]:
# ── LoRA 配置 + 注入 ──────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if MODEL_OK:
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        # 只在 LLM 的注意力/MLP 投影上挂 LoRA；视觉编码器与 connector 不动
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()   # 打印 trainable params / all params 与百分比
else:
    print("[跳过] 基座未加载，无法注入 LoRA。LoRA 可训练参数量的计算见下方练习 2。")


## 4. 演示规模的训练循环（SFT）

最朴素的训练循环，把讲解第 2 节的 SFT 机制跑通：

1. 用 chat 模板得到含回答的输入序列；
2. **`labels = input_ids.clone()`**，并把 padding 位置设为 `-100`（被交叉熵忽略）；
3. 前向 → 模型直接返回 `loss`（标签右移的交叉熵）；反向 → `optimizer.step()`。

> 简化点：这里把整条序列都当标签（含 prompt）。**生产代码必须只对 assistant 回答 token 计 loss**（把 prompt/image 位置也设 `-100`，即讲解里的 label mask）。这里为聚焦 API 而从简，下方 markdown 已注明。

> 显存/时间：4-bit + LoRA + 2B 模型，单张 12GB 卡几分钟跑完这 ~20 个 step。

In [ ]:
# ── 简短训练循环（演示规模）──────────────────────────────────────────
from torch.optim import AdamW

losses = []
if MODEL_OK:
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)

    model.train()
    EPOCHS = 5   # 4 条数据 × 5 = 20 个 step，纯演示

    for epoch in range(EPOCHS):
        for sample in train_samples:
            inputs = build_inputs(sample, with_target=True).to(model.device)

            labels = inputs["input_ids"].clone()
            # 忽略 padding；生产中还需把 prompt/image token 位置也设 -100（label mask）
            labels[labels == processor.tokenizer.pad_token_id] = -100
            inputs["labels"] = labels

            out = model(**inputs)
            loss = out.loss

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            losses.append(loss.item())
            print(f"epoch {epoch} | loss {loss.item():.4f}")

    print("\n训练结束（演示规模）。loss 序列：", [round(x, 3) for x in losses])
else:
    print("[跳过] 基座未加载，跳过 LoRA SFT 训练循环。")


In [ ]:
# ── 画 loss 曲线 ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

if losses:
    plt.figure(figsize=(6, 3))
    plt.plot(losses, marker="o")
    plt.xlabel("step"); plt.ylabel("SFT loss"); plt.title("LoRA SFT loss (demo)")
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
else:
    print("[跳过] 无 loss 数据（训练在无 GPU 时已跳过）。")


## 5. Before / After：对留出样例对比微调前后回答

对一个**没参与训练**的指令生成回答。由于演示数据极小，差异可能细微（甚至看不出明显变化）——
重点是跑通"切换 LoRA adapter 开/关 → 对比生成"的**评测对照流程**。

PEFT 的便利：`with model.disable_adapter():` 临时禁用 LoRA，模型行为回到 4-bit 基座（即"before"）。

In [ ]:
# ── before / after 生成对比 ──────────────────────────────────────────
if MODEL_OK:
    model.eval()
    gen_inputs = build_inputs(heldout_sample, with_target=False).to(model.device)

    @torch.no_grad()
    def generate(inputs):
        out_ids = model.generate(**inputs, max_new_tokens=40, do_sample=False)
        # 只解码新生成的部分
        new_ids = out_ids[:, inputs["input_ids"].shape[1]:]
        return processor.batch_decode(new_ids, skip_special_tokens=True)[0].strip()

    # BEFORE：禁用 LoRA adapter → 等价于原始 4-bit 基座
    with model.disable_adapter():
        before = generate(gen_inputs)

    # AFTER：启用 LoRA adapter → 微调后的模型
    after = generate(gen_inputs)

    print("指令 :", heldout_sample["instruction"])
    print("\n[BEFORE 微调前]:", before)
    print("[AFTER  微调后]:", after)
else:
    print("[跳过] 基座未加载，无法做 before/after 生成对比。")


## 6. DPO 概念演示：用偏好对手算一个 batch 的 DPO loss

把讲解第 4 节的 DPO 落成代码。给定一个偏好对 $(x,\,y_w,\,y_l)$（chosen vs rejected），DPO 损失：

$$ \mathcal{L}_{\text{DPO}} = -\log\sigma\Big(\beta\big[\underbrace{(\log\pi_\theta(y_w)-\log\pi_{\text{ref}}(y_w))}_{r_\theta(x,y_w)/\beta} - (\log\pi_\theta(y_l)-\log\pi_{\text{ref}}(y_l))\big]\Big) $$

其中 $\log\pi(y\mid x)$ 是回答 $y$ 在条件 $x$ 下的**序列对数概率**（只对回答 token 求和）。
**隐式奖励** $r_\theta(x,y)=\beta\log\frac{\pi_\theta(y\mid x)}{\pi_{\text{ref}}(y\mid x)}$——语言模型自己就是奖励模型。

这里 $\pi_\theta$ = 启用 LoRA 的当前模型，$\pi_{\text{ref}}$ = **禁用 LoRA 的基座**（这正是 PEFT 做 DPO 的标准技巧：无需第二份模型权重，靠 `disable_adapter()` 当参考模型）。

我们构造一个偏好对：同一张图、同一指令，**chosen=忠实回答，rejected=幻觉回答（编造了图里没有的物体）**——正对应讲解第 5 节"用偏好抑制 object hallucination"。

In [ ]:
# ── DPO loss 手算（单 batch 偏好对）──────────────────────────────────
import torch.nn.functional as F

pref = {
    "image": CAT_URL,
    "instruction": "Describe the main subjects in this image.",
    "chosen":   "A woman and her dog are sitting together on the beach.",
    "rejected": "A woman, a dog, two surfboards, and a red umbrella are on the beach.",  # 编造了不存在的物体 → 幻觉
}

if MODEL_OK:
    def seq_logprob(response_text):
        '''算回答 response_text 在 (图, 指令) 条件下的序列 log-prob，只对回答 token 求和。'''
        sample = {"image": pref["image"], "instruction": pref["instruction"], "target": response_text}
        full = build_inputs(sample, with_target=True).to(model.device)          # 含回答
        prompt = build_inputs({**sample, "target": ""}, with_target=False).to(model.device)  # 仅 prompt
        prompt_len = prompt["input_ids"].shape[1]   # 回答 token 从这里开始

        logits = model(**full).logits[:, :-1, :]    # 预测下一个 token，错位一格
        labels = full["input_ids"][:, 1:]
        logp = torch.log_softmax(logits.float(), dim=-1)
        token_logp = logp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)  # 每个位置的 log p(真实token)

        # 只累加"回答区间"的 log-prob（prompt_len 之后）
        resp_mask = torch.zeros_like(token_logp)
        resp_mask[:, prompt_len - 1:] = 1.0
        return (token_logp * resp_mask).sum(dim=-1)   # shape [1]

    model.eval()
    with torch.no_grad():
        # π_θ：启用 LoRA（当前策略）
        logp_w_policy = seq_logprob(pref["chosen"])
        logp_l_policy = seq_logprob(pref["rejected"])
        # π_ref：禁用 LoRA（参考策略 = 基座）
        with model.disable_adapter():
            logp_w_ref = seq_logprob(pref["chosen"])
            logp_l_ref = seq_logprob(pref["rejected"])

    beta = 0.1
    # 隐式奖励 r = β·(logπ_θ − logπ_ref)
    r_w = beta * (logp_w_policy - logp_w_ref)   # chosen 的隐式奖励
    r_l = beta * (logp_l_policy - logp_l_ref)   # rejected 的隐式奖励
    dpo_loss = -F.logsigmoid(r_w - r_l)         # -log σ(r_w − r_l)

    print(f"logπ_θ(chosen)   = {logp_w_policy.item():.3f}   logπ_ref(chosen)   = {logp_w_ref.item():.3f}")
    print(f"logπ_θ(rejected) = {logp_l_policy.item():.3f}   logπ_ref(rejected) = {logp_l_ref.item():.3f}")
    print(f"\n隐式奖励 r_chosen   = β·Δlogπ = {r_w.item():.4f}")
    print(f"隐式奖励 r_rejected = β·Δlogπ = {r_l.item():.4f}")
    print(f"奖励 margin (r_w − r_l)        = {(r_w - r_l).item():.4f}")
    print(f"\nDPO loss = -log σ(β(Δlogπ_w − Δlogπ_l)) = {dpo_loss.item():.4f}")
    print("\n解读：margin>0 表示模型已偏好 chosen，loss 小；DPO 训练就是把 margin 推大、"
          "\n      即抬高忠实回答、压低幻觉回答相对参考模型的概率。")
else:
    print("[跳过] 基座未加载，无法手算 DPO loss。")
    print("公式回顾：r = β·(logπ_θ − logπ_ref)；DPO loss = -log σ(r_chosen − r_rejected)，"
          "训练目标是把 chosen 与 rejected 的隐式奖励 margin 推大。")


---
## ✏️ 练习 1：实现 SFT 的 label mask（只对 assistant 回复计 loss）

第 4 节的训练循环为了从简，把整条序列都当标签。现在把"生产代码"该做的事补上：实现 `build_labels(input_ids, segments)`。
`segments` 是 `[(role, length), ...]`，按顺序描述序列里每一段的角色与 token 数（role 取值如 `"system"` / `"image"` / `"user"` / `"assistant"`）。
返回与 `input_ids` 等长的 labels 列表：**`assistant` 段保留原 token id，其余位置一律设为 `-100`**（被交叉熵忽略）。

**提示**：先把 segments 展开成逐 token 的角色列表，再和 `input_ids` 一一配对；若 segments 总长度 `!= len(input_ids)` 应 `raise ValueError`。10 行以内。

In [ ]:
def build_labels(input_ids, segments):
    # TODO: 把 segments 展开成逐 token 的角色列表
    # TODO: 总长度与 len(input_ids) 不一致时 raise ValueError
    # TODO: assistant 位置保留原 token id，其余设为 -100
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ids = [10, 11, 12, 13, 14]
assert build_labels(ids, [("user", 3), ("assistant", 2)]) == [-100, -100, -100, 13, 14]
assert build_labels([1, 2, 3, 4, 5, 6, 7, 8],
                    [("image", 2), ("user", 2), ("assistant", 1), ("user", 1), ("assistant", 2)]) \
       == [-100, -100, -100, -100, 5, -100, 7, 8]        # 多轮对话：只有 assistant 段算 loss
assert build_labels([9, 9], [("user", 2)]) == [-100, -100]   # 没有 assistant 段 → 全部忽略
assert build_labels([], []) == []                            # 边界：空序列
try:
    build_labels([1, 2, 3], [("user", 2)])
    assert False, "长度不匹配应 raise ValueError"
except ValueError:
    pass
print("✅ 练习 1 通过")

## ✏️ 练习 2：手算 LoRA 的可训练参数量与占比

第 3 节 `print_trainable_parameters()` 打印的数字是怎么来的？自己算一遍。实现 `lora_trainable(shapes, r)`：
`shapes` 是被挂 LoRA 的目标矩阵 shape 列表 `[(d_out, d_in), ...]`，`r` 是秩。每个 $W\in\mathbb{R}^{d_{out}\times d_{in}}$
配一对低秩矩阵 $B\in\mathbb{R}^{d_{out}\times r}$、$A\in\mathbb{R}^{r\times d_{in}}$。返回 dict：

- `"trainable"`：LoRA 新增（可训练）参数量；
- `"base"`：原矩阵（冻结）参数量；
- `"percent"`：`trainable / (trainable + base) * 100`，即 `print_trainable_parameters` 里的百分比。

**提示**：每个矩阵的 LoRA 参数 = `r * (d_out + d_in)`；`lora_alpha` 只是缩放系数 $\alpha/r$，**不引入任何参数**。5~8 行。

In [ ]:
def lora_trainable(shapes, r):
    # TODO: trainable = Σ r * (d_out + d_in)
    # TODO: base = Σ d_out * d_in
    # TODO: 返回 {"trainable": ..., "base": ..., "percent": ...}
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
res = lora_trainable([(4, 4)], r=2)
assert res["trainable"] == 16 and res["base"] == 16       # 2*(4+4)=16；玩具矩阵下占比高达 50%
assert abs(res["percent"] - 50.0) < 1e-9

big = lora_trainable([(1536, 1536)], r=8)                 # 真实尺度（Qwen2-VL-2B 的 hidden=1536）
assert big["trainable"] == 8 * (1536 + 1536)
assert big["percent"] < 2.0                               # 大矩阵下 LoRA 只占 ~1%

r8  = lora_trainable([(8, 16), (16, 8)], r=8)
r32 = lora_trainable([(8, 16), (16, 8)], r=32)
assert r32["trainable"] == 4 * r8["trainable"]            # 可训练参数量随 r 线性增长
assert r8["base"] == r32["base"] == 256                   # 冻结的基座参数量与 r 无关
print("✅ 练习 2 通过")

## ✏️ 练习 3：用 numpy 手写 sequence log-prob（DPO 的积木）

第 6 节的 `seq_logprob` 是 DPO 的核心积木。把"错位一格 + log_softmax + 只对回答区间求和"用纯 numpy 复现，不依赖模型。
实现 `seq_logprob_np(logits, ids, prompt_len)`：

- `logits`：shape `(L-1, V)` 的 numpy 数组，`logits[i]` 预测位置 `i+1` 的 token（已错位一格）；
- `ids`：长度 `L` 的完整 token id 列表（prompt + 回答）；
- `prompt_len`：回答 token 从 `ids[prompt_len]` 开始。

返回 `(total, avg)`：回答区间逐 token log-prob 之和，以及**长度归一化**（除以回答 token 数）后的均值。

**提示**：log-softmax 要数值稳定——先减每行 `max` 再算 `log(Σexp)`（即 `x - max - log(Σexp(x - max))`）；
位置 `i+1` 的 token 由 `logits[i]` 预测，所以回答区间对应 `logits[prompt_len - 1:]`。10~15 行。

In [ ]:
import numpy as np

def seq_logprob_np(logits, ids, prompt_len):
    # TODO: 1) 对每行 logits 做数值稳定的 log_softmax
    # TODO: 2) 取 token_logp[i] = logp[i, ids[i+1]]（错位一格）
    # TODO: 3) 只累加回答区间（i >= prompt_len - 1），返回 (total, avg)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
import math
import numpy as np

uni = np.zeros((3, 4))                                    # L=4, V=4：均匀分布，每个 token 概率 1/4
total, avg = seq_logprob_np(uni, [0, 1, 2, 3], prompt_len=2)
assert math.isclose(total, 2 * math.log(0.25), rel_tol=1e-9)   # 回答区间恰好 2 个 token
assert math.isclose(avg, math.log(0.25), rel_tol=1e-9)         # 长度归一化后是单 token 均值

total2, _ = seq_logprob_np(uni + 10.0, [0, 1, 2, 3], prompt_len=2)
assert math.isclose(total, total2, rel_tol=1e-9)          # softmax 平移不变：logits 整体 +10 不变

peaky = np.array([[50.0, 0.0, 0.0, 0.0]])                 # 模型几乎确定下一个 token 是 0
total3, _ = seq_logprob_np(peaky, [9, 0], prompt_len=1)   # ids[0] 是 prompt，ids[1]=0 是回答
assert -1e-6 < total3 <= 0.0                              # log-prob ≈ 0

huge = np.array([[1000.0, 0.0, 0.0, 0.0]])
t4, _ = seq_logprob_np(huge, [9, 0], prompt_len=1)
assert math.isfinite(t4)                                  # 数值稳定：大 logits 不溢出成 nan/inf
print("✅ 练习 3 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def build_labels(input_ids, segments):
    roles = [role for role, length in segments for _ in range(length)]
    if len(roles) != len(input_ids):
        raise ValueError("segments 总长度与 input_ids 不一致")
    return [tid if role == "assistant" else -100
            for tid, role in zip(input_ids, roles)]

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def lora_trainable(shapes, r):
    trainable = sum(r * (d_out + d_in) for d_out, d_in in shapes)
    base = sum(d_out * d_in for d_out, d_in in shapes)
    return {"trainable": trainable, "base": base,
            "percent": trainable / (trainable + base) * 100}

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
import numpy as np

def seq_logprob_np(logits, ids, prompt_len):
    logits = np.asarray(logits, dtype=np.float64)
    m = logits.max(axis=-1, keepdims=True)
    logp = logits - m - np.log(np.exp(logits - m).sum(axis=-1, keepdims=True))  # 稳定 log_softmax
    targets = np.asarray(ids[1:])                          # logits[i] 预测 ids[i+1]
    token_logp = logp[np.arange(len(targets)), targets]
    resp = token_logp[prompt_len - 1:]                     # 只取回答区间
    return float(resp.sum()), float(resp.mean())

## 小结 + 动手练习

**这一章我们跑通了 VLM 第二阶段训练的完整机制**：

- 用 **4-bit (NF4) + LoRA**（即 QLoRA）在单卡上微调 2B VLM —— 可训练参数只占零点几个百分点；
- SFT 训练循环：chat 模板 → label mask → 交叉熵 loss → AdamW step；
- before/after 用 `disable_adapter()` 做对照评测；
- **DPO 概念演示**：用 `π_θ`（启用 LoRA）与 `π_ref`（禁用 LoRA）的序列 log-prob，
  按公式手算隐式奖励 $r=\beta\log(\pi_\theta/\pi_{\text{ref}})$ 与 DPO loss，看清"语言模型即奖励模型"。

**动手练习：**

1. **正确的 label mask**：修改训练循环，把 prompt 与 image placeholder 位置的 label 也设为 `-100`，
   做到"只对 assistant 回答 token 计 loss"。对比一下 loss 曲线和 before/after 生成有何变化。
2. **调 LoRA 秩与 target_modules**：把 `r` 从 8 调到 32（同步 `lora_alpha`），或只挂 `q_proj/v_proj`，
   重新打印 trainable %，观察可训练参数量和 before/after 差异如何变化。
3. **把 DPO 接成训练循环**：把第 6 节的 `dpo_loss` 用一个真实偏好对 batch 反向传播几步
   （记得 `π_ref` 全程 `no_grad` + `disable_adapter`），观察 chosen/rejected 的隐式奖励是否如预期一升一降——
   注意监控两者的**绝对 logprob**，警惕讲解里提到的"二者同时下降"退化。

➡️ **下一站：模块 06 · 高分辨率、任意分辨率与视频**——指令微调让 VLM 会对话，但它还受困于固定低分辨率；
模块 06 讲 AnyRes / NaViT / Qwen2-VL 的 Naive Dynamic Resolution 如何让 VLM 看清小字、长图与视频。

---
## 🎯 真实数据胶囊题：多模态 SFT 的 loss masking（图像 token 不算 loss）

多模态 SFT 里，序列 = [图像 token] + 文本 prompt + 答案；只在**答案**上算 loss——图像 token 和 prompt 都要 mask。用真实图像的 token 数构造序列，验证 mask 正确。

> 本模块新增的**真实数据**练习：用**真实图像**把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, io, urllib.request
import numpy as np
import matplotlib.image as mpimg
CACHE=os.path.expanduser("~/.vlm_data"); os.makedirs(CACHE,exist_ok=True)
def real_image():
    "真实图像 Grace Hopper (来自 matplotlib 示例数据), 返回 (H,W,3) uint8"
    p=os.path.join(CACHE,"grace_hopper.jpg")
    if not os.path.exists(p):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/matplotlib/matplotlib/main/lib/matplotlib/mpl-data/sample_data/grace_hopper.jpg", p)
    return mpimg.imread(p)

img=real_image(); P=14
n_img=(img.shape[0]//P)*(img.shape[1]//P)   # 真实图像 vision token 数
n_prompt=12; n_answer=8
print(f"真实图像 vision token={n_img}, prompt={n_prompt}, answer={n_answer}")

**练习**：实现 `mm_loss_mask(n_img, n_prompt, n_answer)`：返回长 `n_img+n_prompt+n_answer` 的 0/1 数组，只有答案位置为 1（图像 token + prompt 全 0）。

In [ ]:
def mm_loss_mask(n_img, n_prompt, n_answer):
    # TODO: 前 n_img+n_prompt 个为 0，最后 n_answer 个为 1
    raise NotImplementedError


In [ ]:
# 自测
m=mm_loss_mask(n_img, n_prompt, n_answer)
assert len(m)==n_img+n_prompt+n_answer
assert m[:n_img+n_prompt].sum()==0, "图像+prompt 不算 loss"
assert m[n_img+n_prompt:].sum()==n_answer, "只有答案算 loss"
print(f"多模态 masking ✓  {n_img}图+{n_prompt}prompt 全mask, {n_answer}答案计入 loss")


### 📖 参考答案

In [ ]:
def mm_loss_mask(n_img, n_prompt, n_answer):
    m=np.zeros(n_img+n_prompt+n_answer); m[n_img+n_prompt:]=1; return m
print("✓ 图像 token 是条件不是预测目标，必须 mask 掉它的 loss")